# Lezione 7 — Storytelling: i dati non convincono da soli

**Durata stimata: 90 minuti**

## Obiettivi di apprendimento

- Spiegare perché raccontare una storia con i dati è importante quanto analizzarli.
- Trasformare una tabella di risultati in una visualizzazione che comunichi una conclusione chiara.
- Applicare principi base di storytelling dei dati a una presentazione propria.

Questo notebook è autocontenuto: ricarica e pulisce il dataset di ricette (l'hai già fatto nelle lezioni precedenti) e poi costruisce, con codice nuovo, il materiale di questo capitolo. Consulta il [capitolo 7 del manuale](../manual/08_storytelling_IT.md) per la spiegazione completa in prosa.

## 0. Perché un capitolo intero su questo?

Questo tema non è una fase della metodologia di Rollins né di CRISP-DM. Non appare in nessun diagramma delle 10 fasi. Eppure, ha un proprio capitolo perché è l'abilità che la metodologia dà per scontata senza nominarla: comunicare ciò che hai trovato. Si appoggia su tutto il resto — in particolare, sulla visualizzazione, già usata nella Lezione 4 per *esplorare* i dati e che oggi userai per *comunicarli*.

## 1. La frase che riassume il problema

> "Tutti possono mostrare numeri, ma se non hai una storia dietro, se non hai una ragione convincente per agire, alla fine ciò che presenti non si connetterà con il tuo pubblico."

Gli umani capiamo il mondo attraverso **storie**, non attraverso tabelle di numeri. Una tabella di 15 righe e 8 colonne può essere perfettamente corretta e, per la maggior parte delle persone che la ricevono, completamente invisibile.

## 2. Lo studio di Stanford

A Stanford è stato fatto uno studio con due modi di presentare lo stesso pitch (presentazione breve per convincere di qualcosa):

- **Gruppo A**: ha presentato solo KPI (indicatori chiave di prestazione), cifre e statistiche.
- **Gruppo B**: ha presentato quegli stessi dati, ma avvolti in una storia.

Poi è stato chiesto al pubblico cosa ricordasse di ogni presentazione. **Ciò che è rimasto nella memoria sono state le storie.** Le cifre continuavano a essere dentro quelle storie — non che i numeri fossero superflui, è che da soli non venivano ricordati.

**Implicazione pratica:** se presenti una tabella e ti aspetti che la conclusione si ricordi, scommetti contro come funziona la memoria umana. Se la trasformi in una storia chiara, hai molte più probabilità che si traduca in una decisione reale.

## 3. L'ultimo miglio della consegna

*"The last mile in delivery"*: la parte tecnica (caricare, pulire, modellare, valutare) si impara in un periodo di formazione relativamente breve. Ciò che scarseggia è la capacità di **estrarre valore dai dati e comunicarlo** in modo che qualcuno agisca. Per una carriera lunga in questa disciplina, questa è l'abilità che segna la differenza.

E ha un secondo lato: lo storytelling serve anche **all'analista stesso**. Raccontarti una storia su un dataset mentre ci lavori ti aiuta a capirlo meglio — è uno strumento di comprensione, non solo di presentazione finale.

## 4. Semplicità contro complessità: una tensione reale

Raccontare una storia semplice e chiara, senza perdere le sfumature importanti dei dati, **è una tensione difficile, non un problema con trucco**. Più semplifichi, più facile da ricordare — ma più facile che tu nasconda qualcosa di importante. Più sei fedele alla complessità, più onesta la storia — ma meno memorabile.

Non esiste una formula che la risolva una volta per tutte. È un equilibrio che negozi ogni volta che presenti qualcosa, in base a chi ti ascolta e quale decisione è in gioco.

## 5. Da esplorazione a comunicazione

Nella Lezione 4 hai usato tabelle e statistica descrittiva per **esplorare** il dataset di ricette — affinché tu, come analista, capissi cosa c'era lì prima di pulirlo e modellarlo. Questo è l'EDA (analisi esplorativa dei dati): guardare i dati senza pubblico, solo per te.

Oggi userai la visualizzazione con uno scopo diverso: non affinché tu capisca, ma affinché **un'altra persona, senza toccare il dataset, si porti via una conclusione chiara in secondi**. Stesso strumento (un grafico), altro obiettivo.

## 6. Promemoria: pulizia minima del dataset di ricette

Questo l'hai già fatto nelle lezioni precedenti. Lo ripetiamo qui affinché il notebook sia autocontenuto.

In [ ]:
import pandas as pd
import numpy as np

recipes = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv")

column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names
recipes["cuisine"] = recipes["cuisine"].str.lower()

recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

recipes.shape

In [ ]:
# Lo stesso calcolo del capitolo 4: quante ricette contengono ogni ingrediente
ing = recipes.iloc[:, 1:].sum(axis=0)
ingredient = pd.Series(ing.index.values, index=np.arange(len(ing)))
count = pd.Series(list(ing), index=np.arange(len(ing)))
ing_df = pd.DataFrame(dict(ingredient=ingredient, count=count))
ing_df = ing_df[["ingredient", "count"]]
ing_df.sort_values(["count"], ascending=False, inplace=True)
ing_df.reset_index(inplace=True, drop=True)

ing_df.head(10)

## 7. La tabella così com'è: ti ricordi qualcosa?

Guarda la tabella precedente per 10 secondi e poi chiudi gli occhi. Cosa ricordi? Probabilmente "egg" e poco altro, e solo perché è primo. Questo è esattamente il problema descritto dallo studio di Stanford: la tabella è corretta, completa, e non resta nella memoria. Trasformiamo la stessa informazione — senza cambiare un solo numero — in una storia visiva.

## 8. Da tabella a storia: il grafico a barre orizzontale

Un grafico a barre orizzontale è più leggibile di una tabella di testo lunga per diverse ragioni:

- I nomi degli ingredienti sono testo, e in orizzontale si leggono di seguito senza ruotare etichette.
- La lunghezza della barra comunica la grandezza a colpo d'occhio, senza che tu debba confrontare cifre mentalmente.
- Ordinato dal maggiore al minore con il più frequente in cima, l'occhio coglie la conclusione nel primo secondo.

Teniamo i 10 ingredienti più frequenti: è la parte della classifica che davvero interessa a un pubblico, il resto è rumore per questo scopo.

In [ ]:
import matplotlib.pyplot as plt

top10 = ing_df.head(10)

plt.figure(figsize=(8, 5))

# barh() = barre orizzontali. Meglio di un grafico verticale quando
# le etichette di categoria sono testo lungo (nomi di ingredienti).
plt.barh(top10["ingredient"], top10["count"], color="steelblue")

# invert_yaxis() lascia l'ingrediente più frequente in cima:
# è dove l'occhio umano guarda per primo quando legge un grafico.
plt.gca().invert_yaxis()

# Il titolo non descrive il grafico ("Top 10 ingredienti"), dice la
# conclusione direttamente. Questa è la differenza tra tabella e storia.
plt.title("Egg, wheat y butter dominan el dataset (sesgado por ~40.000 recetas americanas)")
plt.xlabel("Número de recetas que lo contienen")
plt.ylabel("Ingrediente")

plt.tight_layout()
plt.show()

Nota: i dati sono identici a quelli del capitolo 4 (egg 21.025, wheat 20.781, butter 20.719...). L'unica cosa che è cambiata è il modo in cui arrivano a chi li guarda. E, seguendo il principio "semplifica senza mentire", il titolo ricorda anche il bias (le ~40.000 ricette americane) invece di nasconderlo — non serve mettere tutto nel grafico, ma nemmeno occultarlo.

## 9. Tabella vs. grafico, fianco a fianco

Affinché tu veda il contrasto d'impatto a colpo d'occhio, ecco la stessa informazione due volte: prima come testo semplice (ciò che vedresti in una tabella dentro un report), e subito sotto, come grafico.

In [ ]:
# Versione "tabella di testo semplice", come apparirebbe in un report senza grafici
print("Ranking de ingredientes más frecuentes (tabla de texto):\n")
for _, row in ing_df.head(10).iterrows():
    print(f"{row['ingredient']:<15} {row['count']:>8.0f}")

In [ ]:
# Stessa informazione, come storia visiva
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10["ingredient"], top10["count"], color="steelblue")
ax.invert_yaxis()
ax.set_title("Los 10 ingredientes más comunes del dataset de recetas")
ax.set_xlabel("Número de recetas que lo contienen")
plt.tight_layout()
plt.show()

Entrambe le celle mostrano esattamente gli stessi 10 ingredienti e gli stessi conteggi. Ma solo una delle due è quella che ricorderesti se le vedessi 5 secondi in una presentazione e passassi alla diapositiva successiva. Questa è l'idea centrale di questo capitolo: lo storytelling non cambia la scoperta, cambia se quella scoperta sopravvive fuori dallo schermo.

## 10. Principi pratici, derivati da tutto quanto sopra

- **Inizia dalla conclusione, non dal metodo.** Il titolo di un grafico dovrebbe dire qualcosa da solo, non descrivere gli assi.
- **Scegli un grafico invece di una tabella quando vuoi che qualcosa si ricordi.** La tabella serve affinché tu esplori; il grafico serve affinché un'altra persona ricordi.
- **Semplifica senza mentire.** Non nascondere la complessità importante (come il bias americano), ma non metterla nemmeno tutta sopra il grafico principale.
- **Dai al tuo pubblico una ragione per agire, non solo informazione.** La differenza tra "questo è ciò che c'è" e "questo è ciò che dovremmo fare" è, spesso, la differenza tra una presentazione che si dimentica e una che cambia una decisione.

## Riepilogo

- Gli umani ricordiamo storie, non tabelle di numeri; lo studio di Stanford lo ha confermato confrontando presentazioni con soli KPI rispetto a presentazioni con storia.
- Lo storytelling è "l'ultimo miglio nella consegna": il tecnico si impara in fretta, comunicare valore è scarso e decisivo per una carriera lunga.
- Esiste una tensione reale, senza soluzione magica, tra semplicità e complessità nel raccontare una storia con i dati.
- Lo storytelling serve anche all'analista stesso per capire meglio il dataset mentre ci lavora.
- La visualizzazione non cambia tra esplorazione (Lezione 4) e comunicazione (questa lezione): cambia lo scopo con cui la usi.
- Trasformare `ing_df` in un `plt.barh()` dei primi 10, con il più frequente in cima e un titolo che dice la conclusione, è la stessa scoperta del capitolo 4 raccontata come storia invece che come tabella.

Nel notebook di pratica applicherai questo stesso principio a una cucina specifica: scegliere i suoi ingredienti caratteristici e trasformarli in un grafico con un titolo che racconti una conclusione, non solo una classifica.